# Five-bus unit commitment and economic dispatch

This notebook evaluates the public MATPOWER `case5` benchmark with a hybrid workflow. The commitment layer is converted to a ten-variable QUBO and solved by the local QAOA implementation. Continuous generator outputs are recovered by a DC-network economic-dispatch problem. An optional local noisy CPUQVM run is included; no cloud or real-device task is submitted.

For each period, the physical balance and unit limits are

$$\sum_g p_{g,t}=D_t,\qquad P_g^{\min}u_{g,t}\le p_{g,t}\le P_g^{\max}u_{g,t}.$$

The QUBO uses the normalized capacity penalty

$$W\left(\sum_g \frac{P_g^{\max}}{S_{\rm base}}u_{g,t}-\frac{D_t+R_t}{S_{\rm base}}\right)^2.$$

In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'src').exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT / 'src'))

from case5_unit_commitment import (
    Case5CommitmentQuboBuilder, Case5QAOA, load_case5_uc,
    solve_milp_uc,
)

## 1. Load the case and inspect the network

The active load is uniformly scaled to 800 MW and 1000 MW over two periods. MATPOWER's 100 MVA base, generator limits, linear costs and branch limits remain unchanged. Startup and no-load costs are small, explicit scenario parameters because the original case5 file does not define unit-commitment costs.

In [ ]:
instance = load_case5_uc()
print('source:', instance.case.source_url)
print('buses:', instance.case.bus_ids)
print('total load:', instance.case.total_load_mw, 'MW')
print('period demands:', instance.demand_mw)
print('PTDF shape:', instance.case.ptdf().shape)
[(unit.name, unit.bus, unit.p_max_mw, unit.linear_cost) for unit in instance.units]

## 2. Classical optimization reference

`solve_milp_uc` uses a mixed-integer linear optimizer. It solves commitment, startup, dispatch, power balance, reserve adequacy, and finite DC line limits jointly. This is the non-enumerative reference.

In [ ]:
classical = solve_milp_uc(instance, evaluate_method='linprog')
print('success:', classical.success)
print('commitments:', classical.commitments)
print('total cost:', classical.schedule.total_cost)
for period, dispatch in enumerate(classical.schedule.dispatch):
    print(period, dispatch.generation_mw, dispatch.branch_flows_mw)

from case5_unit_commitment import evaluate_schedule
slsqp_check = evaluate_schedule(instance, classical.commitments, method='slsqp')
print('SLSQP re-evaluation:', slsqp_check.total_cost)

## 3. QAOA commitment and continuous dispatch

Only commitment bits enter the QUBO, so the ideal local state-vector run uses ten logical qubits. Each capacity-feasible high-probability state is passed to the same `linprog` dispatch routine used to report the classical baseline. Increase `maxiter` for a longer local parameter search.

In [ ]:
qubo = Case5CommitmentQuboBuilder(capacity_weight=200.0).build(instance)
quantum = Case5QAOA(
    instance, backend='statevector', layers=1, seed=11,
    optimizer_options={'options': {'maxiter': 25, 'disp': False}},
    qubo_builder=Case5CommitmentQuboBuilder(capacity_weight=200.0),
).solve(top_k=16, evaluate_method='linprog')
print('logical qubits:', quantum.logical_qubits)
print('capacity-feasible probability:', quantum.feasible_probability)
print('best hybrid cost:', quantum.best_cost)
if quantum.best_candidate is not None:
    print('best commitments:', quantum.best_candidate.commitments)

## 4. Compare and visualize dispatch

The gap is measured against the MILP total cost. A positive value means that the selected QAOA state requires a more expensive commitment schedule; feasibility is checked independently of the QUBO energy.

In [ ]:
gap = None if quantum.best_cost is None else 100 * (quantum.best_cost - classical.schedule.total_cost) / classical.schedule.total_cost
print({'milp_cost': classical.schedule.total_cost, 'qaoa_cost': quantum.best_cost, 'gap_percent': gap})

import matplotlib.pyplot as plt
labels = [unit.name for unit in instance.units]
width = 0.36
x = np.arange(len(labels))
fig, axes = plt.subplots(1, instance.time_periods, figsize=(10, 3.5), sharey=True)
axes = np.atleast_1d(axes)
for period, axis in enumerate(axes):
    axis.bar(x - width/2, classical.schedule.dispatch[period].generation_mw, width, label='MILP')
    if quantum.best_candidate is not None and quantum.best_candidate.schedule is not None:
        axis.bar(x + width/2, quantum.best_candidate.schedule.dispatch[period].generation_mw, width, label='QAOA hybrid')
    axis.set_title(f'period {period}')
    axis.set_xticks(x, labels)
    axis.set_ylabel('MW')
axes[0].legend()
fig.tight_layout()

## 5. Offline noisy FakeBackend rehearsal

The local noisy mode uses pyqpanda3 `NoiseModel` on CPUQVM. It is deliberately synthetic and reproducible; it is not a calibration snapshot of a quantum processor. The same hybrid post-processing can therefore be inspected without an API key:

```python
noisy = Case5QAOA(
    instance, backend='local_noisy', shots=256, layers=1, seed=11,
    optimizer_options={'options': {'maxiter': 4, 'disp': False}},
).solve(top_k=16, evaluate_method='linprog')
print(noisy.execution_environment, noisy.feasible_probability, noisy.best_cost)
```

The optional `scripts/run_fakebackend.py` additionally exercises the OriginQ Runtime `RuntimeService -> QDevice -> FakeBackend` path. Its API-key variable is selected by the caller and its value is never written to results.